# 🤟 Indian Sign Language Translation — Video to Text
**Dataset:** ISL-CSLTR (Kaggle — drblack00)  
**Stack:** Python 3.10 · MediaPipe · PyTorch · Gradio  
**Author:** Your Name

---
### How to use this notebook
Run cells **top to bottom, one section at a time**.  
The notebook is divided into 8 clearly marked sections:

| Section | What it does |
|---|---|
| 1 | Install dependencies |
| 2 | Project folder setup & config |
| 3 | Preprocess videos → extract keypoints |
| 4 | Build PyTorch Dataset & DataLoader |
| 5 | Define model (BiLSTM or Transformer) |
| 6 | Train |
| 7 | Evaluate (accuracy + WER) |
| 8 | Real-time inference + Gradio UI |


## Section 1 — Install dependencies
Run once. Restart kernel after if prompted.

In [ ]:
# Install all required packages
import subprocess, sys

packages = [
    "torch torchvision --index-url https://download.pytorch.org/whl/cpu",  # change to cu121 if you have GPU
    "opencv-python",
    "mediapipe==0.10.11",
    "scikit-learn",
    "tqdm",
    "jiwer",         # Word Error Rate
    "sacrebleu",     # BLEU score
    "gradio",
    "matplotlib",
    "tensorboard",
    "pyyaml",
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", *pkg.split()], check=True)

print("\n✅ All packages installed!")


## Section 2 — Project setup & config
Creates folder structure and sets all hyperparameters in one place.  
**Edit `CFG` below to match your system before proceeding.**


In [ ]:
import os, json, yaml
from pathlib import Path

# ─── All config lives here — edit as needed ───────────────────────────────
CFG = {
    "data": {
        "raw_dir":       "data/raw",        # put your downloaded dataset here
        "processed_dir": "data/processed",
        "max_seq_len":   150,               # frames per video (pad/truncate to this)
        "num_keypoints": 1662,              # 33*4 pose + 468*3 face + 21*3 lh + 21*3 rh
    },
    "model": {
        "type":        "lstm",   # "lstm"  or  "transformer"
        "hidden_size": 256,
        "num_layers":  3,
        "dropout":     0.4,
        "num_heads":   8,        # transformer only
        "ff_dim":      512,      # transformer only
    },
    "training": {
        "epochs":         80,
        "batch_size":     16,
        "learning_rate":  3e-4,
        "patience":       15,    # early stopping
        "label_smoothing":0.1,
        "device":         "cuda",  # set "cpu" if no GPU
    },
    "paths": {
        "checkpoint": "checkpoints/best_model.pt",
        "label_map":  "data/processed/label_map.json",
        "samples":    "data/processed/samples.json",
    }
}

# Create folders
for folder in ["data/raw", "data/processed", "checkpoints", "runs"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

# Save config
with open("configs/config.yaml", "w") as f:
    yaml.dump(CFG, f)

print("✅ Folder structure created:")
print(os.popen("find . -maxdepth 2 -type d | sort").read())


> **Dataset folder structure expected inside `data/raw/`:**
> ```
> data/raw/
> ├── thank_you/
> │   ├── signer1/video1.mp4
> │   ├── signer2/video2.mp4
> ├── how_are_you/
> │   ├── signer1/video1.mp4
> ...
> ```
> Each sub-folder name becomes the **label/class**.


## Section 3 — Preprocessing: Videos → Keypoints
Uses **MediaPipe Holistic** to extract 1662-dimensional keypoints per frame.  
Output: one `.npy` file per video saved in `data/processed/`.  
⏱ *This will take a while depending on dataset size — grab a coffee.*


In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from tqdm.notebook import tqdm
import json, os
from pathlib import Path

mp_holistic = mp.solutions.holistic

RAW  = CFG["data"]["raw_dir"]
PROC = CFG["data"]["processed_dir"]

def extract_keypoints(results):
    """Flatten all MediaPipe landmarks into a single 1662-d vector."""
    pose = np.array([[r.x, r.y, r.z, r.visibility]
                     for r in results.pose_landmarks.landmark]).flatten()            if results.pose_landmarks else np.zeros(33 * 4)

    face = np.array([[r.x, r.y, r.z]
                     for r in results.face_landmarks.landmark]).flatten()            if results.face_landmarks else np.zeros(468 * 3)

    lh   = np.array([[r.x, r.y, r.z]
                     for r in results.left_hand_landmarks.landmark]).flatten()            if results.left_hand_landmarks else np.zeros(21 * 3)

    rh   = np.array([[r.x, r.y, r.z]
                     for r in results.right_hand_landmarks.landmark]).flatten()            if results.right_hand_landmarks else np.zeros(21 * 3)

    return np.concatenate([pose, face, lh, rh])  # shape: (1662,)

# ── Collect all video paths ──
label_map = {}
label_id  = 0
samples   = []

labels_found = sorted([d for d in os.listdir(RAW)
                        if os.path.isdir(os.path.join(RAW, d))])
print(f"Found {len(labels_found)} classes: {labels_found[:10]} ...")

for label in labels_found:
    label_dir = os.path.join(RAW, label)
    if label not in label_map:
        label_map[label] = label_id
        label_id += 1

    video_files = list(Path(label_dir).rglob("*.mp4")) +                   list(Path(label_dir).rglob("*.avi")) +                   list(Path(label_dir).rglob("*.mov"))

    for vpath in tqdm(video_files, desc=f"  '{label}'"):
        frames_kps = []
        cap = cv2.VideoCapture(str(vpath))

        with mp_holistic.Holistic(
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        ) as holistic:
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = holistic.process(rgb)
                frames_kps.append(extract_keypoints(results))
        cap.release()

        if len(frames_kps) == 0:
            print(f"  ⚠ Skipped (no frames): {vpath}")
            continue

        seq       = np.array(frames_kps, dtype=np.float32)  # (T, 1662)
        stem      = vpath.stem
        save_path = os.path.join(PROC, f"{label}__{stem}.npy")
        np.save(save_path, seq)
        samples.append({"path": save_path, "label": label})

# ── Save metadata ──
json.dump(label_map, open(CFG["paths"]["label_map"], "w"), indent=2)
json.dump(samples,   open(CFG["paths"]["samples"],   "w"), indent=2)

print(f"\n✅ Done! {len(samples)} sequences | {len(label_map)} classes")
print(f"   Label map: {CFG['paths']['label_map']}")


### Quick sanity check — visualize one sample

In [ ]:
import matplotlib.pyplot as plt

# Load a sample and plot keypoint activity over time
sample     = samples[0]
seq        = np.load(sample["path"])
label_name = sample["label"]

plt.figure(figsize=(14, 3))
plt.imshow(seq.T, aspect="auto", cmap="viridis", interpolation="nearest")
plt.colorbar(label="Keypoint value")
plt.xlabel("Frame (time)")
plt.ylabel("Keypoint dimension (1662)")
plt.title(f"Keypoint heatmap — label: '{label_name}'  |  {seq.shape[0]} frames")
plt.tight_layout()
plt.show()

print(f"Sequence shape : {seq.shape}")
print(f"Value range    : {seq.min():.3f} → {seq.max():.3f}")


## Section 4 — Dataset & DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

MAX_LEN = CFG["data"]["max_seq_len"]

class ISLDataset(Dataset):
    def __init__(self, samples, label_map):
        self.samples   = samples
        self.label_map = label_map

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        seq = np.load(s["path"])     # (T, 1662)
        T   = seq.shape[0]

        # Pad or truncate to MAX_LEN
        if T >= MAX_LEN:
            seq  = seq[:MAX_LEN]
            mask = np.ones(MAX_LEN, dtype=np.float32)
        else:
            pad  = np.zeros((MAX_LEN - T, seq.shape[1]), dtype=np.float32)
            seq  = np.concatenate([seq, pad], axis=0)
            mask = np.array([1.0] * T + [0.0] * (MAX_LEN - T), dtype=np.float32)

        label = self.label_map[s["label"]]
        return (
            torch.tensor(seq,   dtype=torch.float32),
            torch.tensor(mask,  dtype=torch.float32),
            torch.tensor(label, dtype=torch.long),
        )

# ── Load saved metadata ──
samples_all = json.load(open(CFG["paths"]["samples"]))
label_map   = json.load(open(CFG["paths"]["label_map"]))
id2label    = {v: k for k, v in label_map.items()}

labels_all  = [s["label"] for s in samples_all]

# ── Stratified split: 70/15/15 ──
train_s, temp_s = train_test_split(samples_all, test_size=0.30,
                                   stratify=labels_all, random_state=42)
temp_labels = [s["label"] for s in temp_s]
val_s, test_s   = train_test_split(temp_s, test_size=0.50,
                                   stratify=temp_labels, random_state=42)

BS = CFG["training"]["batch_size"]
train_loader = DataLoader(ISLDataset(train_s, label_map), batch_size=BS, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(ISLDataset(val_s,   label_map), batch_size=BS, shuffle=False, num_workers=2)
test_loader  = DataLoader(ISLDataset(test_s,  label_map), batch_size=BS, shuffle=False, num_workers=2)

print(f"✅ Train: {len(train_s)} | Val: {len(val_s)} | Test: {len(test_s)} samples")
print(f"   Classes: {len(label_map)}")
print(f"   Batch shape demo:")
seqs, masks, labels = next(iter(train_loader))
print(f"     seqs  → {seqs.shape}   (batch, frames, keypoints)")
print(f"     masks → {masks.shape}")
print(f"     labels→ {labels.shape}")


## Section 5 — Model definition
Two architectures are provided. Set `CFG["model"]["type"]` to `"lstm"` or `"transformer"`.  
**Start with `"lstm"` — it trains faster on small datasets.**


In [ ]:
import torch.nn as nn
import math

# ════════════════════════════════════════════════════════════
# Model A: BiLSTM with attention  (recommended for starters)
# ════════════════════════════════════════════════════════════
class ISL_BiLSTM(nn.Module):
    def __init__(self, input_size=1662, num_classes=100):
        super().__init__()
        h  = CFG["model"]["hidden_size"]
        L  = CFG["model"]["num_layers"]
        dp = CFG["model"]["dropout"]

        self.input_norm = nn.LayerNorm(input_size)
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=h,
            num_layers=L,
            batch_first=True,
            bidirectional=True,
            dropout=dp if L > 1 else 0.0,
        )
        self.attn    = nn.Linear(h * 2, 1)
        self.dropout = nn.Dropout(dp)
        self.fc      = nn.Linear(h * 2, num_classes)

    def forward(self, x, mask=None):
        x = self.input_norm(x)
        out, _ = self.lstm(x)                            # (B, T, 2H)
        scores  = self.attn(out).squeeze(-1)             # (B, T)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)  # (B, T, 1)
        context = (out * weights).sum(dim=1)             # (B, 2H)
        return self.fc(self.dropout(context))


# ════════════════════════════════════════════════════════════
# Model B: Transformer encoder
# ════════════════════════════════════════════════════════════
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200):
        super().__init__()
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class ISL_Transformer(nn.Module):
    def __init__(self, input_size=1662, num_classes=100):
        super().__init__()
        d  = CFG["model"]["hidden_size"]
        h  = CFG["model"]["num_heads"]
        ff = CFG["model"]["ff_dim"]
        L  = CFG["model"]["num_layers"]
        dp = CFG["model"]["dropout"]

        self.proj    = nn.Linear(input_size, d)
        self.pe      = PositionalEncoding(d, max_len=MAX_LEN + 10)
        enc_layer    = nn.TransformerEncoderLayer(d_model=d, nhead=h,
                           dim_feedforward=ff, dropout=dp, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=L)
        self.pool    = nn.AdaptiveAvgPool1d(1)
        self.fc      = nn.Linear(d, num_classes)

    def forward(self, x, mask=None):
        x = self.pe(self.proj(x))
        key_mask = (mask == 0) if mask is not None else None
        out = self.encoder(x, src_key_padding_mask=key_mask)   # (B, T, d)
        out = self.pool(out.transpose(1, 2)).squeeze(-1)        # (B, d)
        return self.fc(out)


# ── Instantiate based on config ──
DEVICE      = torch.device(CFG["training"]["device"] if torch.cuda.is_available() else "cpu")
NUM_CLASSES = len(label_map)
INPUT_SIZE  = CFG["data"]["num_keypoints"]

if CFG["model"]["type"] == "transformer":
    model = ISL_Transformer(input_size=INPUT_SIZE, num_classes=NUM_CLASSES).to(DEVICE)
else:
    model = ISL_BiLSTM(input_size=INPUT_SIZE, num_classes=NUM_CLASSES).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model  : {model.__class__.__name__}")
print(f"   Device : {DEVICE}")
print(f"   Params : {total_params:,}")
print()
print(model)


## Section 6 — Training

In [ ]:
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import os

LR       = CFG["training"]["learning_rate"]
EPOCHS   = CFG["training"]["epochs"]
PATIENCE = CFG["training"]["patience"]
LS       = CFG["training"]["label_smoothing"]
CKPT     = CFG["paths"]["checkpoint"]

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=LS)

history = {"train_loss": [], "train_acc": [], "val_acc": []}
best_val_acc, patience_ctr = 0.0, 0
os.makedirs("checkpoints", exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch:03d} [train]", leave=False)
    for seqs, masks, lbls in loop:
        seqs, masks, lbls = seqs.to(DEVICE), masks.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        out  = model(seqs, masks)
        loss = criterion(out, lbls)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        correct    += (out.argmax(1) == lbls).sum().item()
        total      += lbls.size(0)
        loop.set_postfix(loss=f"{loss.item():.4f}")

    train_acc  = correct / total
    avg_loss   = total_loss / len(train_loader)

    # ── Validate ───────────────────────────────────────────
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for seqs, masks, lbls in val_loader:
            seqs, masks, lbls = seqs.to(DEVICE), masks.to(DEVICE), lbls.to(DEVICE)
            out = model(seqs, masks)
            val_correct += (out.argmax(1) == lbls).sum().item()
            val_total   += lbls.size(0)
    val_acc = val_correct / val_total
    scheduler.step()

    history["train_loss"].append(avg_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:03d} | loss {avg_loss:.4f} | train {train_acc:.3f} | val {val_acc:.3f}", end="")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_ctr = 0
        torch.save({"model_state": model.state_dict(),
                    "label_map":   label_map,
                    "config":      CFG}, CKPT)
        print("  ← best ✓")
    else:
        patience_ctr += 1
        print(f"  (patience {patience_ctr}/{PATIENCE})")
        if patience_ctr >= PATIENCE:
            print("\n⏹ Early stopping triggered.")
            break

print(f"\n✅ Training complete. Best val accuracy: {best_val_acc:.4f}")


### Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history["train_loss"], label="Train loss", color="#3B82F6")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history["train_acc"], label="Train accuracy", color="#10B981")
axes[1].plot(history["val_acc"],   label="Val accuracy",   color="#F59E0B")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()


## Section 7 — Evaluation on test set

In [ ]:
from jiwer import wer as compute_wer
import matplotlib.pyplot as plt
import numpy as np

# ── Load best checkpoint ──
ckpt = torch.load(CKPT, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()

refs, hyps = [], []
with torch.no_grad():
    for seqs, masks, lbls in tqdm(test_loader, desc="Evaluating"):
        seqs, masks = seqs.to(DEVICE), masks.to(DEVICE)
        out   = model(seqs, masks)
        preds = out.argmax(1).cpu().tolist()
        for pred, true in zip(preds, lbls.tolist()):
            hyps.append(id2label[pred])
            refs.append(id2label[true])

correct = sum(h == r for h, r in zip(hyps, refs))
acc     = correct / len(refs)
word_er = compute_wer(refs, hyps)

print(f"{'='*45}")
print(f"  Test accuracy    : {acc:.4f}  ({correct}/{len(refs)} correct)")
print(f"  Word error rate  : {word_er:.4f}")
print(f"{'='*45}")

# ── Per-class accuracy bar chart ──
from collections import defaultdict
class_correct = defaultdict(int)
class_total   = defaultdict(int)
for h, r in zip(hyps, refs):
    class_total[r]   += 1
    class_correct[r] += int(h == r)

classes     = sorted(class_total.keys())
class_accs  = [class_correct[c] / class_total[c] for c in classes]

plt.figure(figsize=(max(10, len(classes)*0.4), 4))
bars = plt.bar(classes, class_accs, color="#6366F1", alpha=0.85)
plt.axhline(acc, color="#EF4444", linestyle="--", label=f"Overall {acc:.2f}")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.ylabel("Accuracy")
plt.title("Per-class accuracy on test set")
plt.ylim(0, 1.1)
plt.legend()
plt.tight_layout()
plt.savefig("per_class_accuracy.png", dpi=150)
plt.show()


### Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

ref_ids  = [label_map[r] for r in refs]
hyp_ids  = [label_map[h] for h in hyps]
cm       = confusion_matrix(ref_ids, hyp_ids)
class_names = [id2label[i] for i in range(NUM_CLASSES)]

fig, ax = plt.subplots(figsize=(max(8, NUM_CLASSES*0.35), max(6, NUM_CLASSES*0.35)))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(class_names, rotation=90, fontsize=7)
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(class_names, fontsize=7)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion matrix")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()


## Section 8 — Inference
Two options: **8A** for a single video file, **8B** for live webcam, **8C** for Gradio web UI.


### 8A — Predict from a video file

In [ ]:
def predict_video(video_path, model, id2label, device, max_len=MAX_LEN):
    """Run inference on a single video file and return ranked predictions."""
    cap = cv2.VideoCapture(str(video_path))
    kps_list = []
    with mp_holistic.Holistic(min_detection_confidence=0.5) as holistic:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(rgb)
            kps_list.append(extract_keypoints(results))
    cap.release()

    if not kps_list:
        return None, "No frames detected in video."

    seq = np.array(kps_list[:max_len], dtype=np.float32)
    T   = seq.shape[0]
    if T < max_len:
        seq = np.concatenate([seq, np.zeros((max_len - T, seq.shape[1]))], axis=0)

    seq_t  = torch.tensor(seq).unsqueeze(0).to(device)
    mask_t = torch.ones(1, max_len).to(device)

    model.eval()
    with torch.no_grad():
        out    = model(seq_t, mask_t)
        probs  = out.softmax(1)[0]
        top5   = probs.topk(min(5, NUM_CLASSES))

    top_label = id2label[out.argmax(1).item()]
    ranked    = [(id2label[i.item()], f"{s.item()*100:.1f}%")
                 for s, i in zip(top5.values, top5.indices)]
    return top_label, ranked

# ── Usage: change the path to any video file ──
VIDEO_PATH = "data/raw/thank_you/signer1/sample.mp4"   # ← edit this

if os.path.exists(VIDEO_PATH):
    top, ranked = predict_video(VIDEO_PATH, model, id2label, DEVICE)
    print(f"\n🏆 Predicted: {top}")
    print("\nTop predictions:")
    for label, conf in ranked:
        print(f"  {label:<30} {conf}")
else:
    print(f"File not found: {VIDEO_PATH}\nEdit VIDEO_PATH above to point to a real video.")


### 8B — Live webcam inference
Press **Q** to quit the window.

In [ ]:
# Live webcam — run in a Python script if OpenCV window doesn't show inside Jupyter
# Alternatively use: %matplotlib notebook with cv2_imshow for Colab

def run_webcam(model, id2label, device, max_len=MAX_LEN, stride=10):
    """Sliding-window real-time inference from webcam."""
    model.eval()
    cap = cv2.VideoCapture(0)
    sequence    = []
    prediction  = "Waiting..."
    confidence  = 0.0

    with mp_holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as holistic:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(rgb)
            kps     = extract_keypoints(results)
            sequence.append(kps)

            if len(sequence) == max_len:
                seq_arr = np.array(sequence, dtype=np.float32)
                seq_t   = torch.tensor(seq_arr).unsqueeze(0).to(device)
                mask_t  = torch.ones(1, max_len).to(device)
                with torch.no_grad():
                    out = model(seq_t, mask_t)
                probs      = out.softmax(1)[0]
                pred_id    = out.argmax(1).item()
                prediction = id2label[pred_id]
                confidence = probs[pred_id].item()
                sequence   = sequence[stride:]      # slide window

            # Draw hand landmarks
            mp_draw = mp.solutions.drawing_utils
            mp_draw.draw_landmarks(frame, results.left_hand_landmarks,
                                   mp.solutions.holistic.HAND_CONNECTIONS)
            mp_draw.draw_landmarks(frame, results.right_hand_landmarks,
                                   mp.solutions.holistic.HAND_CONNECTIONS)

            # Display prediction overlay
            bar_w = int(confidence * 300)
            cv2.rectangle(frame, (10, 60), (10 + bar_w, 80), (0, 200, 100), -1)
            cv2.rectangle(frame, (10, 60), (310, 80), (200, 200, 200), 1)
            cv2.putText(frame, f"Prediction : {prediction}",
                        (10, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 230, 100), 2)
            cv2.putText(frame, f"Confidence : {confidence*100:.1f}%",
                        (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (230, 230, 230), 1)
            cv2.putText(frame, f"Buffer     : {len(sequence)}/{max_len}",
                        (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)

            cv2.imshow("ISL Translation  |  Press Q to quit", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()

run_webcam(model, id2label, DEVICE)


### 8C — Gradio web UI
Run the cell below — a link will appear to open the UI in your browser.

In [ ]:
import gradio as gr

def gradio_predict(video_path):
    if video_path is None:
        return "No video uploaded."
    top, ranked = predict_video(video_path, model, id2label, DEVICE)
    result  = f"🏆  **Predicted:** {top}\n\n"
    result += "**Top 5 predictions:**\n"
    for lbl, conf in ranked:
        result += f"- `{lbl}` — {conf}\n"
    return result

demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Video(label="Upload ISL sign video"),
    outputs=gr.Markdown(label="Translation result"),
    title="🤟 Indian Sign Language Translator",
    description=(
        "Upload a video clip of an ISL sign. "
        "The model predicts the sentence class and shows top-5 confidence scores."
    ),
    examples=[],
    theme=gr.themes.Soft(),
)

demo.launch(share=False, inbrowser=True)
# Set share=True to get a public URL (useful if running on a remote server)


---
## Next steps & tips

| Goal | What to do |
|---|---|
| Better accuracy | Increase `epochs`, reduce `learning_rate`, add data augmentation |
| Switch to Transformer | Set `CFG["model"]["type"] = "transformer"` in Section 2, re-run Sections 5–7 |
| Add more data | Download INCLUDE dataset (IIT Madras) and merge with ISL-CSLTR |
| GPU training | Set `CFG["training"]["device"] = "cuda"` — needs NVIDIA GPU + CUDA toolkit |
| Monitor training | Run `tensorboard --logdir runs/` in terminal while training |
| Export model | `torch.onnx.export(model, ...)` for deployment |

> 💡 **Tip:** Run Section 3 only once — re-running overwrites all `.npy` files. If you add new videos, simply re-run Section 3.
